# BigEarthNet to Petastorm-Compatible Parquet

This notebook:
1. Reads metadata from S3
2. Stratified samples by split (train/val/test)
3. Loads S2, S1, and reference map images
4. Writes to Parquet format (readable by Petastorm)

In [ ]:
from pyspark.sql import SparkSession


spark = SparkSession.builder \
    .appName("Fractal") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.access.key", "") \
    .config("spark.hadoop.fs.s3a.secret.key", "") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "50000") \
    .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60000") \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "30000") \
    .config("spark.hadoop.fs.s3a.multipart.purge.age", "30000000") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/12 16:53:34 WARN Utils: Your hostname, Omowonuolas-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.2.190.9 instead (on interface en0)
26/01/12 16:53:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/omowonuolaakintola/anaconda3/envs/petastorm_env/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/omowonuolaakintola/.ivy2.5.2/cache
The jars for the packages stored in: /Users/omowonuolaakintola/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7cc164a1-ddd6-4f3a-8726-d473cd24f2cf;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.1 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.901 in central
	found org.wildfly.openssl#wildfly-op

In [5]:
import boto3
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import rasterio
from rasterio.enums import Resampling
from io import BytesIO
import json

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, BinaryType

## Configuration

In [13]:
# S3 paths
BUCKET = 'ubs-homes'
METADATA_KEY = 'erasmus/ola/bigearthnet/metadata/metadata_updated.parquet'
OUTPUT_PATH = 's3://ubs-homes/erasmus/ola/bigearthnet/petastorm_dataset'

# Band configuration
S2_BANDS = [ 'B02', 'B03', 'B04' 'B08']
S1_BANDS = ['VV', 'VH']
TARGET_SIZE = (120, 120)

# Sampling config
SUBSET_ENABLED = True
SUBSET_FRACTION = 0.0001      # 1% of data
SUBSET_SEED = 42

## Define Schema

In [14]:
output_schema = StructType([
    StructField('patch_id', StringType(), False),
    StructField('split', StringType(), False),
    StructField('labels', StringType(), False),      # JSON string of label list
    StructField('s2_image', BinaryType(), False),    # (120, 120, 12) as bytes
    StructField('s1_image', BinaryType(), False),    # (120, 120, 2) as bytes
    StructField('label_map', BinaryType(), False),   # (120, 120) as bytes
])

## Helper Functions

In [15]:
# ===== S3 Client (global) =====
s3 = boto3.client('s3')


# ===== Helper Functions =====
def parse_s3_path(s3_path):
    path = s3_path.replace("s3://", "")
    bucket = path.split("/")[0]
    key = "/".join(path.split("/")[1:])
    return bucket, key


def load_band_from_s3(s3_path, target_size=(120, 120)):
    bucket, key = parse_s3_path(s3_path)
    response = s3.get_object(Bucket=bucket, Key=key)  # Uses global s3
    file_bytes = BytesIO(response['Body'].read())
    
    with rasterio.open(file_bytes) as src:
        data = src.read(
            1,
            out_shape=target_size,
            resampling=Resampling.bilinear
        )
    return data.astype(np.float32)


def load_s2_patch(s2_folder_path, patch_id):
    bands = []
    for band in S2_BANDS:
        band_path = f"{s2_folder_path}{patch_id}_{band}.tif"
        bands.append(load_band_from_s3(band_path, TARGET_SIZE))
    return np.stack(bands, axis=-1)


def load_s1_patch(s1_folder_path, s1_name):
    bands = []
    for band in S1_BANDS:
        band_path = f"{s1_folder_path}{s1_name}_{band}.tif"
        bands.append(load_band_from_s3(band_path, TARGET_SIZE))
    return np.stack(bands, axis=-1)


def load_reference_map(ref_folder_path, patch_id):
    ref_path = f"{ref_folder_path}{patch_id}_reference_map.tif"
    return load_band_from_s3(ref_path, TARGET_SIZE)

## Stratified Sampling

In [16]:
def stratified_sample_by_split(df, frac=0.01, seed=42):
    """Stratified sample by split (train/val/test)"""
    splits = df.select("split").distinct().rdd.flatMap(lambda x: x).collect()
    fractions = {s: frac for s in splits}
    sampled_df = df.stat.sampleBy("split", fractions, seed)
    
    print(f"\n===== Stratified Sample Summary =====")
    print(f"Original size: {df.count():,}")
    print(f"Sampled size: {sampled_df.count():,}")
    print("\nBy split:")
    sampled_df.groupBy("split").count().show()
    
    return sampled_df

## Process Partition Function

In [7]:
def process_partition(iterator):
    """Process each partition - loads images and yields rows"""
    # Create S3 client inside partition (can't serialize boto3 client)
    s3_client = boto3.client('s3')
    
    for row in iterator:
        try:
            # Load images
            s2_image = load_s2_patch(s3_client, row.s2_path, row.patch_id)
            s1_image = load_s1_patch(s3_client, row.s1_path, row.s1_name)
            label_map = load_reference_map(s3_client, row.ref_path, row.patch_id)
            
            # Yield as tuple matching schema
            yield (
                str(row.patch_id),
                str(row.split),
                json.dumps(list(row.labels)),     # Convert list to JSON string
                bytes(s2_image.tobytes()),        # Serialize as bytes
                bytes(s1_image.tobytes()),
                bytes(label_map.tobytes()),
            )
        except Exception as e:
            print(f"Error processing {row.patch_id}: {e}")
            continue

## Load Metadata

In [17]:
# Load metadata from S3
print(f"Loading metadata from s3://{BUCKET}/{METADATA_KEY}")
s3 = boto3.client('s3')
response = s3.get_object(Bucket=BUCKET, Key=METADATA_KEY)
metadata_df = pq.read_table(BytesIO(response['Body'].read())).to_pandas()

print(f"Loaded {len(metadata_df)} rows")
print(f"Columns: {metadata_df.columns.tolist()}")
metadata_df.head()

Loading metadata from s3://ubs-homes/erasmus/ola/bigearthnet/metadata/metadata_updated.parquet
Loaded 480038 rows
Columns: ['patch_id', 'labels', 'split', 'country', 's1_name', 's2v1_name', 'contains_seasonal_snow', 'contains_cloud_or_shadow', 's1_path', 's2_path', 'ref_path']


,patch_id,labels,split,country,s1_name,s2v1_name,contains_seasonal_snow,contains_cloud_or_shadow,s1_path,s2_path,ref_path
0,S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_2...,"[Arable land, Broad-leaved forest, Mixed fores...",test,Austria,S1B_IW_GRDH_1SDV_20170612T165809_33UUP_26_57,S2A_MSIL2A_20170613T101031_26_57,False,False,s3://ubs-datasets/bigearthnet/BigEarthNet-S1/S...,s3://ubs-datasets/bigearthnet/BigEarthNet-S2/S...,s3://ubs-datasets/bigearthnet/Reference_Maps/S...
1,S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_2...,"[Arable land, Broad-leaved forest, Inland wate...",test,Austria,S1B_IW_GRDH_1SDV_20170612T165809_33UUP_27_55,S2A_MSIL2A_20170613T101031_27_55,False,False,s3://ubs-datasets/bigearthnet/BigEarthNet-S1/S...,s3://ubs-datasets/bigearthnet/BigEarthNet-S2/S...,s3://ubs-datasets/bigearthnet/Reference_Maps/S...
2,S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_2...,"[Arable land, Broad-leaved forest, Coniferous ...",test,Austria,S1B_IW_GRDH_1SDV_20170612T165809_33UUP_27_56,S2A_MSIL2A_20170613T101031_27_56,False,False,s3://ubs-datasets/bigearthnet/BigEarthNet-S1/S...,s3://ubs-datasets/bigearthnet/BigEarthNet-S2/S...,s3://ubs-datasets/bigearthnet/Reference_Maps/S...
3,S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_2...,"[Broad-leaved forest, Complex cultivation patt...",test,Austria,S1B_IW_GRDH_1SDV_20170612T165809_33UUP_27_57,S2A_MSIL2A_20170613T101031_27_57,False,False,s3://ubs-datasets/bigearthnet/BigEarthNet-S1/S...,s3://ubs-datasets/bigearthnet/BigEarthNet-S2/S...,s3://ubs-datasets/bigearthnet/Reference_Maps/S...
4,S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_2...,"[Broad-leaved forest, Complex cultivation patt...",test,Austria,S1B_IW_GRDH_1SDV_20170612T165809_33UUP_27_58,S2A_MSIL2A_20170613T101031_27_58,False,False,s3://ubs-datasets/bigearthnet/BigEarthNet-S1/S...,s3://ubs-datasets/bigearthnet/BigEarthNet-S2/S...,s3://ubs-datasets/bigearthnet/Reference_Maps/S...


## Sample

In [18]:
# Use pandas
# Stratified sample with Pandas
if SUBSET_ENABLED:
    metadata_df = metadata_df.groupby('split', group_keys=False).apply(
        lambda x: x.sample(frac=SUBSET_FRACTION, random_state=SUBSET_SEED)
    )
    print(f"Sampled {len(metadata_df)} rows")
    print(metadata_df['split'].value_counts())

# Convert labels to JSON string
#metadata_df['labels'] = metadata_df['labels'].apply(json.dumps)

Sampled 48 rows
split
train         24
test          12
validation    12
Name: count, dtype: int64


/var/folders/2m/h3tmqc297_qfhms3m1j64ygc0000gn/T/ipykernel_49361/2621238699.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  metadata_df = metadata_df.groupby('split', group_keys=False).apply(


26/01/12 18:31:26 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 266188 ms exceeds timeout 120000 ms
26/01/12 18:31:26 WARN SparkContext: Killing executors is not supported by current scheduler.
26/01/12 18:44:21 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1363)
	at o

## Process and Write to Parquet

In [10]:
#3. Process patches
records = []
total = len(metadata_df)

for idx, row in metadata_df.iterrows():
    try:
        s2_image = load_s2_patch(row['s2_path'], row['patch_id'])
        s1_image = load_s1_patch(row['s1_path'], row['s1_name'])
        label_map = load_reference_map(row['ref_path'], row['patch_id'])
        
        records.append({
            'patch_id': row['patch_id'],
            'split': row['split'],
            'labels': json.dumps(list(row['labels'])),
            's2_image': s2_image.tobytes(),
            's1_image': s1_image.tobytes(),
            'label_map': label_map.tobytes(),
        })
        
        if len(records) % 100 == 0:
            print(f"Processed {len(records)}/{total}...")
            
    except Exception as e:
        print(f"Error {row['patch_id']}: {e}")
        continue

# 4. Save as Parquet
print(f"\nSaving {len(records)} records to {OUTPUT_PATH}")
output_df = pd.DataFrame(records)
#output_df.to_parquet(OUTPUT_PATH, index=False)
#print("Done!")

Processed 100/4800...
Processed 200/4800...
Processed 300/4800...
Processed 400/4800...
Processed 500/4800...
Processed 600/4800...
Processed 700/4800...
Processed 800/4800...
Processed 900/4800...
Processed 1000/4800...
Processed 1100/4800...
Processed 1200/4800...
Processed 1300/4800...
Processed 1400/4800...
Processed 1500/4800...
Processed 1600/4800...
Processed 1700/4800...
Processed 1800/4800...
Processed 1900/4800...
Processed 2000/4800...
Processed 2100/4800...
Processed 2200/4800...
Processed 2300/4800...
Processed 2400/4800...
Processed 2500/4800...
Processed 2600/4800...
Processed 2700/4800...
Processed 2800/4800...
Processed 2900/4800...
Processed 3000/4800...
Processed 3100/4800...
Processed 3200/4800...
Processed 3300/4800...
Processed 3400/4800...
Processed 3500/4800...
Processed 3600/4800...
Processed 3700/4800...
Processed 3800/4800...
Processed 3900/4800...
Processed 4000/4800...
Processed 4100/4800...
Processed 4200/4800...
Processed 4300/4800...
Processed 4400/4800.

In [12]:
output_df.to_parquet('petastorm_dataset.parquet', index=False)
print(f"Saved locally: {len(output_df)} records")

# Upload to S3
s3.upload_file(
    'petastorm_dataset.parquet',
    'ubs-homes',
    'erasmus/ola/bigearthnet/petastorm_dataset.parquet'
)
print("Uploaded to S3!")

Saved locally: 4800 records
Uploaded to S3!


## Verify Output

In [14]:
import pandas as pd
import boto3
from io import BytesIO

# Read from S3 - remove this part
s3 = boto3.client('s3')
response = s3.get_object(
    Bucket='ubs-homes',
    Key='erasmus/ola/bigearthnet/petastorm_dataset.parquet'
)

output_df = pd.read_parquet(BytesIO(response['Body'].read()))

print(f"Total rows: {len(output_df)}")
print(output_df[['patch_id', 'split', 'labels']].head())

Total rows: 4800
                                            patch_id split  \
0  S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_3...  test   
1  S2B_MSIL2A_20170930T095019_N9999_R079_T34UEG_5...  test   
2  S2A_MSIL2A_20171002T094031_N9999_R036_T34TCR_5...  test   
3  S2A_MSIL2A_20170905T095031_N9999_R079_T35VNL_3...  test   
4  S2B_MSIL2A_20180515T094029_N9999_R036_T35VNJ_5...  test   

                                              labels  
0  ["Arable land", "Complex cultivation patterns"...  
1  ["Arable land", "Coniferous forest", "Land pri...  
2                                    ["Arable land"]  
3  ["Arable land", "Coniferous forest", "Mixed fo...  
4  ["Coniferous forest", "Land principally occupi...  


In [15]:
print(output_df.head())

                                            patch_id split  \
0  S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_3...  test   
1  S2B_MSIL2A_20170930T095019_N9999_R079_T34UEG_5...  test   
2  S2A_MSIL2A_20171002T094031_N9999_R036_T34TCR_5...  test   
3  S2A_MSIL2A_20170905T095031_N9999_R079_T35VNL_3...  test   
4  S2B_MSIL2A_20180515T094029_N9999_R036_T35VNJ_5...  test   

                                              labels  \
0  ["Arable land", "Complex cultivation patterns"...   
1  ["Arable land", "Coniferous forest", "Land pri...   
2                                    ["Arable land"]   
3  ["Arable land", "Coniferous forest", "Mixed fo...   
4  ["Coniferous forest", "Land principally occupi...   

                                            s2_image  \
0  b'\x00\x00~C\x00\x00;C\x00\x80\xb7C\x00\x00nC\...   
1  b'\x00\x00\x0cC\x00\x80\xa1C\x00\xc01D\x00\x00...   
2  b'\x00@3D\x00\x80.D\x00\xc0dD\x00\x80\x93D\x00...   
3  b'\x00\x00=C\x00\x00\x15C\x00\x00\x9bC\x00\x00...   
4  b'\x00\

In [16]:
import numpy as np
import json

# Get first row
row = output_df.iloc[0]

# Decode arrays
s2_image = np.frombuffer(row['s2_image'], dtype=np.float32).reshape(120, 120, 12)
s1_image = np.frombuffer(row['s1_image'], dtype=np.float32).reshape(120, 120, 2)
label_map = np.frombuffer(row['label_map'], dtype=np.float32).reshape(120, 120)
labels = json.loads(row['labels'])

# Verify
print(f"Patch ID: {row['patch_id']}")
print(f"Labels: {labels}")
print(f"\nS2 shape: {s2_image.shape}")
print(f"S2 dtype: {s2_image.dtype}")
print(f"S2 min/max: {s2_image.min():.2f} / {s2_image.max():.2f}")
print(f"\nS1 shape: {s1_image.shape}")
print(f"S1 min/max: {s1_image.min():.2f} / {s1_image.max():.2f}")
print(f"\nLabel map shape: {label_map.shape}")
print(f"Label unique values: {np.unique(label_map)}")

Patch ID: S2A_MSIL2A_20170613T101031_N9999_R022_T33UUP_39_66
Labels: ['Arable land', 'Complex cultivation patterns', 'Inland wetlands', 'Land principally occupied by agriculture, with significant areas of natural vegetation', 'Mixed forest', 'Pastures']

S2 shape: (120, 120, 12)
S2 dtype: float32
S2 min/max: 111.00 / 6111.00

S1 shape: (120, 120, 2)
S1 min/max: -30.62 / 0.98

Label map shape: (120, 120)
Label unique values: [211. 231. 242. 243. 313. 411.]


26/01/09 20:15:14 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 913850 ms exceeds timeout 120000 ms
26/01/09 20:15:14 WARN SparkContext: Killing executors is not supported by current scheduler.
26/01/09 20:28:17 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:669)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1296)
	at o